# 05 — Validation

Benchmarks: rate matching, action-potential comparison, resting potential,
stochastic convergence, and singularity handling.

In [1]:
import sys; sys.path.insert(0, '/workspace')
import os, warnings; warnings.filterwarnings('ignore')
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['font.family'] = ['Liberation Sans','Arimo','DejaVu Sans']
matplotlib.rcParams['svg.fonttype'] = 'none'
FIG_DIR = '/mnt/results/hh_simulator/figures'
os.makedirs(FIG_DIR, exist_ok=True)
from hh_simulator import (presets, NaChannel, KChannel, LeakChannel,
    PointCell, Simulator, step_pulse, analysis, viz)
V = np.linspace(-100, 50, 601)

In [2]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for p, c in zip(('m','h','n'), ('#0279EE','#FD9BED','#FF9400')):
    a,b = presets.CLASSIC_RATES[p]
    el = presets.fitted_landscape(p)
    xinf_c = a(V)/(a(V)+b(V)); tau_c = 1.0/(a(V)+b(V))
    axes[0].plot(V, el.x_inf(V), color=c, lw=1.5, label=f'{p} Eyring')
    axes[0].plot(V, xinf_c, color=c, lw=1, ls=':', label=f'{p} classic')
    axes[1].plot(V, el.tau(V), color=c, lw=1.5)
    axes[1].plot(V, tau_c, color=c, lw=1, ls=':')
axes[0].set_xlabel('V (mV)'); axes[0].set_ylabel('x_inf')
axes[0].legend(frameon=False, fontsize=8)
axes[1].set_xlabel('V (mV)'); axes[1].set_ylabel('tau (ms)')
fig.suptitle('Rate matching: Eyring (solid) vs classic HH (dotted)')
fig.tight_layout()
fig.savefig(f'{FIG_DIR}/05_rate_matching.svg', bbox_inches='tight')
fig.savefig(f'{FIG_DIR}/05_rate_matching.png', bbox_inches='tight', dpi=150)
plt.show()

In [3]:
t_eval = np.linspace(0, 40, 4001)
I_fn = step_pulse((0,40), 10.0, onset=5, dur=30)
cell_c = PointCell([NaChannel('classic'), KChannel('classic'), LeakChannel()])
cell_e = PointCell([NaChannel('energy'), KChannel('energy'), LeakChannel()])
sol_c = Simulator(cell_c).run((0,40), I_inj=I_fn, mode='deterministic', t_eval=t_eval)
sol_e = Simulator(cell_e).run((0,40), I_inj=I_fn, mode='deterministic', t_eval=t_eval)
fig, ax = plt.subplots(figsize=(7,3.2))
ax.plot(t_eval, sol_c.V, 'k-', lw=1.2, label='classic')
ax.plot(t_eval, sol_e.V, '#E9ED4C', lw=1.2, ls='--', label='energy')
ax.set_xlabel('Time (ms)'); ax.set_ylabel('Voltage (mV)')
ax.set_title(f'AP: classic peak={np.max(sol_c.V):.1f} mV, energy peak={np.max(sol_e.V):.1f} mV')
ax.legend(frameon=False)
fig.savefig(f'{FIG_DIR}/05_ap_comparison.svg', bbox_inches='tight')
fig.savefig(f'{FIG_DIR}/05_ap_comparison.png', bbox_inches='tight', dpi=150)
plt.show()

In [4]:
rmsds = []; Ns = [1000, 3000, 10000, 30000, 100000]
for N in Ns:
    rng = np.random.default_rng(7)
    s = Simulator(cell_c).run((0,40), I_inj=I_fn, mode='stochastic', dt=0.01,
        N_channels={'Na':N,'K':N//3}, rng=rng, record_every=5)
    Vi = np.interp(t_eval, s.t, s.V)
    sub = sol_c.V < -30
    rmsds.append(np.sqrt(np.mean((Vi[sub]-sol_c.V[sub])**2)))
fig, ax = plt.subplots(figsize=(5,3.5))
ax.loglog(Ns, rmsds, 'o-', color='#0279EE', lw=1.5)
ax.set_xlabel('N (Na channels)'); ax.set_ylabel('Subthreshold RMSD (mV)')
ax.set_title('Stochastic convergence to deterministic')
fig.savefig(f'{FIG_DIR}/05_convergence.svg', bbox_inches='tight')
fig.savefig(f'{FIG_DIR}/05_convergence.png', bbox_inches='tight', dpi=150)
plt.show()

In [5]:
print('=== Validation Summary ===')
print(f'Resting potential (classic): {cell_c.resting_potential():.3f} mV')
print(f'Resting potential (energy):  {cell_e.resting_potential():.3f} mV')
print(f'AP peak (classic): {np.max(sol_c.V):.2f} mV')
print(f'AP peak (energy):  {np.max(sol_e.V):.2f} mV')
print(f'alpha_n(-55) = {float(presets.alpha_n(-55.0)):.4f} (singularity OK)')
print(f'alpha_m(-40) = {float(presets.alpha_m(-40.0)):.4f} (singularity OK)')
for p in ('m','h','n'):
    a,b = presets.CLASSIC_RATES[p]
    xinf_c = a(V)/(a(V)+b(V))
    el = presets.fitted_landscape(p)
    print(f'{p}: x_inf max err = {np.max(np.abs(el.x_inf(V)-xinf_c)):.4f}')


=== Validation Summary ===
Resting potential (classic): -65.000 mV
Resting potential (energy):  -65.089 mV
AP peak (classic): 40.27 mV
AP peak (energy):  39.42 mV
alpha_n(-55) = 0.1000 (singularity OK)
alpha_m(-40) = 1.0000 (singularity OK)
m: x_inf max err = 0.0162
h: x_inf max err = 0.0109
n: x_inf max err = 0.0538
